<a href="https://colab.research.google.com/github/tymoteuszdatta/AleAgenci_scout/blob/main/AA_90minut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO

In [7]:
# POBIERANIE KADRY
def pobierz_kadre_90minut(url_skarb):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"Pobieram dane z adresu: {url_skarb}")
    response = requests.get(url_skarb, headers=headers)
    response.encoding = 'iso-8859-2'

    soup = BeautifulSoup(response.text, 'html.parser')

    # Sprawdźmy, czy w ogóle pobrało poprawną stronę
    tytul_strony = soup.find('title')
    if tytul_strony:
        print(f"Tytuł pobranej strony: {tytul_strony.text.strip()}")

    zawodnicy = []
    # Sprawdzajmy różne warianty linków na 90minut (czasem to 'kariera.php?id=', a czasem podstrony klubowe)
    for link in soup.find_all('a', href=True):
        href = link['href']
        if 'kariera.php?id=' in href:
            imie_nazwisko = link.text.strip()
            if imie_nazwisko and len(imie_nazwisko) > 2: # odrzucamy pojedyncze znaczki
                zawodnicy.append({
                    'Zawodnik': imie_nazwisko,
                    'ID_90minut': href.split('id=')[1].split('&')[0],
                    'Profil': f"http://www.90minut.pl/{href}" if not href.startswith('http') else href
                })

    df = pd.DataFrame(zawodnicy).drop_duplicates(subset=['ID_90minut'])
    print(f"Znaleziono unikalnych zawodników: {len(df)}")
    return df

In [43]:
def pobierz_bilans_90minut(url_bilans):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print(f"Pobieram dane bilansu z: {url_bilans}")
    response = requests.get(url_bilans, headers=headers)
    response.encoding = 'iso-8859-2'

    html_io = StringIO(response.text)
    tabele = pd.read_html(html_io)
    print(f"Znaleziono wszystkich tabel na stronie: {len(tabele)}")

    tabela_wynikowa = None

    # Szukamy tabeli, która wygląda jak właściwa kadra (ma więcej niż 5 wierszy)
    for i, t in enumerate(tabele):
        # Sprawdzamy rozmiar tabeli (wiersze, kolumny)
        print(f"Tabela nr {i}: wymiary {t.shape}")
        if t.shape[0] > 5:  # szukamy tabeli z dłuższą listą zawodników
            tabela_wynikowa = t
            print(f"-> Wybieram tabelę numer {i} jako właściwy bilans zawodników.")
            break

    # Jeśli żadna nie spełnia warunku, bierzemy największą
    if tabela_wynikowa is None and len(tabele) > 0:
        tabela_wynikowa = max(tabele, key=lambda x: x.shape[0])

    df_bilans = tabela_wynikowa

    if df_bilans is not None:
      print(f"Sukces! Pobrano tabelę o wymiarach: {df_bilans.shape}")
      # Wyświetlamy pierwsze wiersze
    df_czysty = df_bilans.copy()
    df_czysty.columns = df_czysty.iloc[2]
    df_czysty = df_czysty.iloc[3:].reset_index(drop=True)
    df_czysty = df_czysty.dropna(axis=1, how='all')
    df_czysty = df_czysty.loc[:, df_czysty.columns.notna()]
    df_czysty = df_czysty.iloc[:-2].reset_index(drop=True)
    print("Sformatowana tabela zawodników:")

    return df_czysty

In [44]:
url_bilans = "http://www.90minut.pl/bilans.php?id_klub=42&id_sezon=109"
pobierz_bilans_90minut(url_bilans)

Pobieram dane bilansu z: http://www.90minut.pl/bilans.php?id_klub=42&id_sezon=109
Znaleziono wszystkich tabel na stronie: 3
Tabela nr 0: wymiary (2, 4)
Tabela nr 1: wymiary (4, 1)
Tabela nr 2: wymiary (26, 12)
-> Wybieram tabelę numer 2 jako właściwy bilans zawodników.
Sukces! Pobrano tabelę o wymiarach: (26, 12)
Sformatowana tabela zawodników:


2,imię i nazwisko,w.,pełne,z ławki,czas,ż.k.,cz.k.,br.,sam.,k+,k-
0,Hubert Adamczyk,14,2,4,858,0,0,0,0,0,0
1,Mateusz Bąkowicz,14,10,0,1207,4,1,1,0,0,0
2,Bartosz Borowski,1,0,0,65,0,0,0,0,0,0
3,Maciej Famulak,13,1,7,505,1,0,0,0,0,0
4,Konrad Janicki,12,0,7,542,2,0,5,0,0,0
5,Dmytro Juchymowycz,11,9,0,919,2,0,0,0,0,0
6,Filip Mosek,11,0,1,690,0,0,2,0,0,0
7,Ramil Mustafajew,6,1,4,220,0,0,0,0,0,0
8,Damian Nowacki,14,9,4,1003,5,0,1,0,0,0
9,Jakub Oleksiewicz,14,0,11,393,0,0,4,0,0,0


In [10]:
url_testowy = "http://www.90minut.pl/kadra.php?id_sezon=109&jesien=1&id_klub=42" #TESTOWY LINK DO WKLEJENIA

In [11]:
df_klubu = pobierz_kadre_90minut(url_testowy)
display(df_klubu.head(10))

Pobieram dane z adresu: http://www.90minut.pl/kadra.php?id_sezon=109&jesien=1&id_klub=42
Tytuł pobranej strony: Kadra - Chojniczanka Chojnice
Znaleziono unikalnych zawodników: 29


,Zawodnik,ID_90minut,Profil
0,Maksymilian Bydler,53592,http://www.90minut.pl//kariera.php?id=53592
1,Adrian Czerniewicz,51286,http://www.90minut.pl//kariera.php?id=51286
2,Damian Primel,14535,http://www.90minut.pl//kariera.php?id=14535
3,Mateusz Bąkowicz,36423,http://www.90minut.pl//kariera.php?id=36423
4,Jakub Goliński,45585,http://www.90minut.pl//kariera.php?id=45585
5,Dmytro Juchymowycz,44912,http://www.90minut.pl//kariera.php?id=44912
6,Mateusz Morek,52653,http://www.90minut.pl//kariera.php?id=52653
7,Ramil Mustafajew,42005,http://www.90minut.pl//kariera.php?id=42005
8,Jan Smolarczyk,50382,http://www.90minut.pl//kariera.php?id=50382
9,Dawid Szot,36238,http://www.90minut.pl//kariera.php?id=36238


In [ ]:
----------------------------------------------------------------------------------------------------------------------------------------------------

In [25]:
df_czysty = df_bilans.copy()
df_czysty.columns = df_czysty.iloc[2]
df_czysty = df_czysty.iloc[3:].reset_index(drop=True)
df_czysty = df_czysty.dropna(axis=1, how='all')
df_czysty = df_czysty.loc[:, df_czysty.columns.notna()]

In [26]:
print("Sformatowana tabela zawodników:")
display(df_czysty.head(15))

Sformatowana tabela zawodników:


2,imię i nazwisko,w.,pełne,z ławki,czas,ż.k.,cz.k.,br.,sam.,k+,k-
0,Hubert Adamczyk,14,2,4,858,0,0,0,0,0,0
1,Mateusz Bąkowicz,14,10,0,1207,4,1,1,0,0,0
2,Bartosz Borowski,1,0,0,65,0,0,0,0,0,0
3,Maciej Famulak,13,1,7,505,1,0,0,0,0,0
4,Konrad Janicki,12,0,7,542,2,0,5,0,0,0
5,Dmytro Juchymowycz,11,9,0,919,2,0,0,0,0,0
6,Filip Mosek,11,0,1,690,0,0,2,0,0,0
7,Ramil Mustafajew,6,1,4,220,0,0,0,0,0,0
8,Damian Nowacki,14,9,4,1003,5,0,1,0,0,0
9,Jakub Oleksiewicz,14,0,11,393,0,0,4,0,0,0


In [12]:
df_baza = df_klubu.copy()

In [13]:
df_baza['Status_Skautingowy'] = 'Do weryfikacji'

In [14]:
display(df_baza[['Zawodnik', 'Profil', 'Status_Skautingowy']].head(15))

,Zawodnik,Profil,Status_Skautingowy
0,Maksymilian Bydler,http://www.90minut.pl//kariera.php?id=53592,Do weryfikacji
1,Adrian Czerniewicz,http://www.90minut.pl//kariera.php?id=51286,Do weryfikacji
2,Damian Primel,http://www.90minut.pl//kariera.php?id=14535,Do weryfikacji
3,Mateusz Bąkowicz,http://www.90minut.pl//kariera.php?id=36423,Do weryfikacji
4,Jakub Goliński,http://www.90minut.pl//kariera.php?id=45585,Do weryfikacji
5,Dmytro Juchymowycz,http://www.90minut.pl//kariera.php?id=44912,Do weryfikacji
6,Mateusz Morek,http://www.90minut.pl//kariera.php?id=52653,Do weryfikacji
7,Ramil Mustafajew,http://www.90minut.pl//kariera.php?id=42005,Do weryfikacji
8,Jan Smolarczyk,http://www.90minut.pl//kariera.php?id=50382,Do weryfikacji
9,Dawid Szot,http://www.90minut.pl//kariera.php?id=36238,Do weryfikacji
